In [5]:
import pandas as pd
df=pd.read_csv("../data/processed_sales_data.csv",parse_dates=["Date"])


In [6]:
df["Date"]=pd.to_datetime(df["Date"])
df["Year"]=df["Date"].dt.year
df["Month"]=df["Date"].dt.month
df["Day"]=df["Date"].dt.day
df["Week"]=df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"]=df["Date"].dt.quarter
df["DayOfWeek"]=df["Date"].dt.day_of_week
df["IsWeekend"]=(df["DayOfWeek"]>=5).astype(int)

df["IsMonthStart"]=df["Date"].dt.is_month_start.astype(int)
df["IsMonthEnd"]=df["Date"].dt.is_month_end.astype(int)

# Encode Categorical Variable

In [7]:
df["StateHoliday"]=df["StateHoliday"].astype(str)
df=pd.get_dummies(df,columns=["StateHoliday"],drop_first=True)


# Lag Features


In [8]:
# Previous sales of the SAME store
df["Lag_1"] = (
    df.groupby("Store")["Sales"]
      .shift(1)
)

# Sales 7 observations earlier for SAME store
df["Lag_7"] = (
    df.groupby("Store")["Sales"]
      .shift(7)
)

# Sales 30 observations earlier for SAME store
df["Lag_30"] = (
    df.groupby("Store")["Sales"]
      .shift(30)
)

# Rolling Features
IMPORTANT:
shift(1) ensures that current day's Sales

 NOT used in the rolling calculation.
 
 This prevents target leakage.


In [9]:
df["RollingMean7"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

df["RollingMean30"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(30).mean()
      )
)


# ==========================================
# 7. Store-wise Rolling Standard Deviation
# ==========================================

df["RollingStd7"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(7).std()
      )
)

df["RollingStd30"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(30).std()
      )
)

# Expanding Mean

In [10]:
df["ExpandingMean"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).expanding().mean()
      )
)


# Remove NaN Values

In [11]:
df.dropna(inplace=True)


# Reset Index

In [12]:
df.reset_index(drop=True,inplace=True)


# Final Check

In [13]:
print("\nFirst 5 Rows:")
print(df.head())

print("\nColumns:")
print(df.columns.tolist())

print("\nDataset Information:")
print(df.info())




First 5 Rows:
   Store  DayOfWeek       Date  Sales  Customers  Open  Promo  SchoolHoliday  \
0      1          2 2013-02-06   6140        693     1      1              0   
1      1          3 2013-02-07   5499        675     1      1              0   
2      1          4 2013-02-08   5681        630     1      1              0   
3      1          5 2013-02-09   5370        656     1      0              0   
4      1          0 2013-02-11   4409        599     1      0              0   

   Year  Month  ...  StateHoliday_b  StateHoliday_c   Lag_1   Lag_7  Lag_30  \
0  2013      2  ...           False           False  6049.0  3725.0  5530.0   
1  2013      2  ...           False           False  6140.0  4601.0  4327.0   
2  2013      2  ...           False           False  5499.0  4709.0  4486.0   
3  2013      2  ...           False           False  5681.0  5633.0  4997.0   
4  2013      2  ...           False           False  5370.0  5970.0  7176.0   

   RollingMean7  RollingMean3

# Save feature-engineered dataset


In [14]:
df.to_csv("featured__new_sales_data.csv", index=False)

print("Feature-engineered dataset saved successfully!")



Feature-engineered dataset saved successfully!
